In [ ]:
# =============================================================================
# CHANDIPUR PXE TIDE & RANGE SAFETY PORTAL - EDA & MODEL TRAINING NOTEBOOK
# =============================================================================

# 1. IMPORT REQUIRED LIBRARIES
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMRegressor, LGBMClassifier
from sklearn.metrics import r2_score, mean_squared_error, accuracy_score

In [ ]:
# Styling for plots
sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (10, 5)

In [ ]:
# =============================================================================
# 2. DATA LOADING FROM SQLITE DATABASE
# =============================================================================
db_path = os.path.abspath("../data/pxe_tide_portal.db")
print(f"Connecting to database at: {db_path}")

In [ ]:
conn = sqlite3.connect(db_path)
df = pd.read_sql("SELECT * FROM tide_telemetry", conn)
conn.close()

In [ ]:
print("Dataset Initial Shape:", df.shape)
print(df.head())

In [ ]:
# =============================================================================
# 3. DATA CLEANING & EXPLORATORY DATA ANALYSIS (EDA)
# =============================================================================
print("\n--- Missing Values Check ---")
print(df.isnull().sum())

In [ ]:
print("\n--- Statistical Summary ---")
print(df.describe())

In [ ]:
# A. Correlation Matrix Heatmap
plt.figure(figsize=(8, 6))
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Chandipur Telemetry Feature Correlation Matrix", fontsize=14, color='white')
plt.show()

In [ ]:
# B. Distribution of Tide Gauge Levels
plt.figure(figsize=(8, 4))
sns.histplot(df['main_gauge_level_m'], kde=True, color='skyblue', bins=30)
plt.title("Distribution of Main Tide Gauge Levels (Meters)")
xlabel = plt.xlabel("Gauge Level (m)")
plt.show()

In [ ]:
# C. Wind Speed vs Gauge Level Scatter Plot
plt.figure(figsize=(8, 4))
sns.scatterplot(x='wind_speed_ms', y='main_gauge_level_m', data=df, alpha=0.5, color='orange')
plt.title("Wind Speed vs Tide Gauge Level")
plt.show()

In [ ]:
# =============================================================================
# 4. FEATURE TRANSFORMATION & PREPROCESSING
# =============================================================================
# Drop non-feature columns for machine learning
target_column = "main_gauge_level_m"
drop_cols = ['id', 'timestamp', target_column]

In [ ]:
X = df.drop(columns=[col for col in drop_cols if col in df.columns])
y_reg = df[target_column]

In [ ]:
# Create safety classification target (1 = Safe to fire if gauge < 3.0m and wind < 12m/s, else 0)
y_clf = ((y_reg < 3.0) & (X['wind_speed_ms'] < 12.0)).astype(int)

In [ ]:
# Train-Test Split
X_train, X_test, y_train_reg, y_test_reg, y_train_clf, y_test_clf = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42
)

In [ ]:
print(f"\nTraining Data Shape: {X_train.shape}")
print(f"Testing Data Shape: {X_test.shape}")

In [ ]:
# Feature Scaling using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Save Preprocessor Artifact
os.makedirs("../artifacts", exist_ok=True)
joblib.dump(scaler, "../artifacts/preprocessor.pkl")
print("Preprocessor object successfully saved to artifacts/preprocessor.pkl")

In [ ]:
# =============================================================================
# 5. MODEL TRAINING & EVALUATION METRICS
# =============================================================================

# --- A. LightGBM Regressor (Tide Level Prediction) ---
reg_model = LGBMRegressor(n_estimators=100, learning_rate=0.05, random_state=42)
reg_model.fit(X_train_scaled, y_train_reg)

In [ ]:
reg_preds = reg_model.predict(X_test_scaled)
r2 = r2_score(y_test_reg, reg_preds)
rmse = mean_squared_error(y_test_reg, reg_preds, squared=False)

In [ ]:
print(f"\n[LightGBM Regressor Metrics]")
print(f"R2 Score : {r2:.4f}")
print(f"RMSE     : {rmse:.4f}")

In [ ]:
joblib.dump(reg_model, "../artifacts/tide_level_model.joblib")

In [ ]:
# --- B. LightGBM Classifier (Firing Range Safety Gate) ---
clf_model = LGBMClassifier(n_estimators=100, learning_rate=0.05, random_state=42)
clf_model.fit(X_train_scaled, y_train_clf)

In [ ]:
clf_preds = clf_model.predict(X_test_scaled)
accuracy = accuracy_score(y_test_clf, clf_preds)

In [ ]:
print(f"\n[LightGBM Classifier Metrics]")
print(f"Accuracy : {accuracy:.4f}")

In [ ]:
joblib.dump(clf_model, "../artifacts/testing_gate_model.joblib")
print("\nAll models and preprocessors successfully trained and exported!")